## 介紹

這堂課將涵蓋：
- 什麼是函式呼叫及其使用情境
- 如何使用 OpenAI 建立函式呼叫
- 如何將函式呼叫整合到應用程式中

## 學習目標

完成本課程後，您將知道如何並了解：

- 使用函式呼叫的目的
- 使用 OpenAI 服務設定函式呼叫
- 設計適合您應用程式使用情境的有效函式呼叫


## 理解函式呼叫 

在這堂課中，我們想為我們的教育新創公司建立一個功能，讓使用者可以使用聊天機器人來尋找技術課程。我們將推薦符合他們技能水平、目前職務和感興趣技術的課程。 

為了完成這個目標，我們將結合使用： 
 - `OpenAI` 來為使用者創建聊天體驗
 - `Microsoft Learn Catalog API` 來協助使用者根據其需求尋找課程 
 - `函式呼叫` 將使用者的查詢傳送給函式以執行 API 請求。 

讓我們先從為什麼要使用函式呼叫開始了解： 


### 為什麼使用函式呼叫

如果你已經完成了本課程的其他課程，你應該能理解使用大型語言模型（LLMs）的強大功能。希望你也能看到它們的一些限制。

函式呼叫是 OpenAI 服務的一項功能，旨在解決以下挑戰：

回應格式不一致：
- 在引入函式呼叫之前，大型語言模型的回應是無結構且不一致的。開發者必須編寫複雜的驗證代碼來處理輸出中的各種變化。

與外部資料整合有限：
- 在此功能推出之前，將應用程式其他部分的資料併入聊天語境中非常困難。

透過標準化回應格式並啟用與外部資料的無縫整合，函式呼叫簡化了開發並減少了額外驗證邏輯的需求。

使用者無法獲得像「斯德哥爾摩目前的天氣如何？」這樣的答案。因為模型僅限於訓練資料的時間範圍。

讓我們看看下面說明此問題的範例：

假設我們想建立一個學生資料庫，以便向他們建議適合的課程。下面有兩個學生的描述，在包含的資料上非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想要將此發送給大型語言模型（LLM）來解析資料。這可以稍後在我們的應用程式中使用，將其發送到 API 或存儲在資料庫中。

讓我們建立兩個相同的提示，指示 LLM 我們感興趣的資訊：


我們想將這個發送給大型語言模型（LLM），以解析對我們產品重要的部分。因此我們可以創建兩個相同的提示來指導LLM：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


在建立這兩個提示後，我們將使用 `client.responses.create` 將它們傳送到大型語言模型 (LLM)。我們將提示儲存在 `input` 變數中，並將角色指派為 `user`。這是為了模擬使用者向聊天機器人發送訊息的情形。 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-4o-mini"


: 

現在我們可以將這兩個請求一起發送到 LLM，並檢查我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



即使提示相同且描述類似，我們仍然可能得到不同格式的 `Grades` 屬性。

如果你多次執行上述儲存格，格式可能是 `3.7` 或 `3.7 GPA`。

這是因為 LLM 接收的是以書寫提示形式的不結構化資料，輸出也同樣是不結構化資料。我們需要有結構化的格式，這樣在儲存或使用這些資料時才知道預期的內容。

透過使用功能呼叫，我們可以確保收到結構化的資料回傳。在使用功能呼叫時，LLM 並不會實際呼叫或執行任何函式。而是我們建立一個結構供 LLM 遵循其回應格式。接著利用這些結構化回應來判斷該在應用程式中執行哪個函式。
 


![函式呼叫流程圖](../../../../translated_images/zh-TW/Function-Flow.083875364af4f4bb.webp)


我們可以將函式回傳的結果取出並將其傳回給 LLM。LLM 將接著使用自然語言來回答使用者的查詢。 


### 函式呼叫的使用案例

<strong>呼叫外部工具</strong>
聊天機器人在提供使用者問題答案方面表現優異。透過使用函式呼叫，聊天機器人可以利用使用者的訊息來完成特定任務。例如，學生可以請聊天機器人「寄信給我的老師說我需要在這個科目上得到更多協助」。這可以透過呼叫函式 `send_email(to: string, body: string)` 來完成


**建立 API 或資料庫查詢**
使用者可以用自然語言找到資訊，並將其轉換成格式化的查詢或 API 請求。舉例來說，一位老師可以請求「哪些學生完成了最後一個作業」，這可能會呼叫一個名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函式


<strong>建立結構化資料</strong>
使用者可以將一段文字或 CSV 檔案交給大型語言模型來擷取其中的重要資訊。例如，學生可以轉換一篇關於和平協議的維基百科文章，來創建 AI 快閃卡。這可以透過呼叫函式 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 來實現


## 2. 建立您的第一個函數呼叫

建立函數呼叫的過程包含三個主要步驟：
1. 使用函數列表和使用者訊息呼叫 Chat Completions API
2. 讀取模型的回應以執行動作，例如執行函數或 API 呼叫
3. 再次呼叫 Chat Completions API，並將您的函數回應傳入，以利用該資訊產生對使用者的回應。


![函數呼叫流程](../../../../translated_images/zh-TW/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素

#### 使用者輸入

第一步是建立一個使用者訊息。這可以透過取得文字輸入的值動態指派，或者您也可以在此處指定一個值。如果這是您第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（終端使用者）。對於函式呼叫，我們將這裡設為 `user` 並給出一個示範問題。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。

接下來我們將定義一個函式及該函式的參數。我們這裡只會使用一個名為 `search_courses` 的函式，但你可以建立多個函式。

<strong>重要</strong>：函式會包含在系統訊息中傳送給 LLM，並且會計入你可用的 token 數量。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

函數定義結構有多層級，各自擁有不同屬性。以下是巢狀結構的說明：

**頂層函數屬性：**

`name` - 想要被呼叫的函數名稱。

`description` - 函數運作方式的描述。在此描述中，清晰且具體非常重要。

`parameters` - 一份模型在回應中應產生的值與格式的清單。

**參數物件屬性：**

`type` - 參數物件的資料類型（通常為 "object"）

`properties` - 模型將用於回應的具體值清單。

**個別參數屬性：**

`name` - 由屬性鍵隱含定義（例如 "role"、"product"、"level"）

`type` - 該特定參數的資料類型（例如 "string"、"number"、"boolean"）

`description` - 該特定參數的描述

**選用屬性：**

`required` - 一個列出為完成函數呼叫必須提供的參數的陣列


### 呼叫函式  
定義函式之後，我們現在需要將其包含在呼叫 Chat Completion API 中。我們通過在請求中添加 `functions` 來做到這一點。在這個例子中為 `functions=functions`。  

也可以選擇將 `function_call` 設定為 `auto`。這表示我們讓大型語言模型根據使用者訊息決定應該呼叫哪一個函式，而不是自行指派。  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們來看回應，看看它是如何格式化的：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函式名稱被呼叫了，且根據使用者訊息，LLM 能找到適合函式參數的資料。


## 3.將函數調用整合到應用程式中。


在測試完來自 LLM 的格式化回應後，我們現在可以將其整合到應用程式中。

### 流程管理

要將此整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 OpenAI 服務並將訊息儲存在名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函式，用來呼叫 Microsoft Learn API 以取得課程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為一種最佳實踐，我們將接著查看模型是否想要呼叫一個函式。之後，我們會建立可用函式之一並將其對應到正在被呼叫的函式。 
接著，我們將取得函式的參數並將它們映射到從 LLM 傳出的參數。

最後，我們會附加函式呼叫訊息以及由 `search_courses` 訊息回傳的值。這讓 LLM 擁有所有需要的資訊
以自然語言回應使用者。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將發送更新後的訊息給 LLM，這樣我們就可以收到自然語言回應，而不是 API JSON 格式的回應。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代碼挑戰 

很棒的工作！要繼續學習 OpenAI 函數調用，你可以建立： https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 函數的更多參數可能有助於學習者找到更多課程。你可以在這裡找到可用的 API 參數： 
 - 建立另一個函數調用，從學習者那裡取得更多資訊，例如他們的母語 
 - 在函數調用和/或 API 呼叫未返回任何合適課程時，建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
